In [1]:
# -*- coding: utf-8 -*-
"""
Qiskit Digital Lesson for Google Colab: Superposition, Entanglement, and BB84

This script is designed to be run in a Google Colab or Jupyter Notebook environment.
It provides a purely digital, simulation-based introduction to quantum computing concepts.

It covers:
1.  Superposition: Demonstrating a qubit's ability to be in multiple states.
2.  Entanglement: Exploring the "spooky action at a distance" of Bell states.
3.  Quantum Key Distribution (BB84): Simulating a secure communication
    protocol, including an eavesdropper scenario.

Instructions:
Run each cell sequentially by clicking the "play" button next to it.
"""

# CELL 1: INSTALLATION
# This command installs the necessary Python libraries in the Colab environment.
# The '-q' flag makes the output less verbose.
!pip install qiskit qiskit-aer matplotlib -q
print("✅ Libraries installed successfully!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 88.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.0/109.0 kB 6.7 MB/s eta 0:00:00
✅ Libraries installed successfully!


In [2]:
# CELL 2: IMPORTS
# Now we import the functions we'll need from the installed libraries.
import random
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


In [3]:
# CELL 3: PART 1 - SUPERPOSITION
def demonstrate_superposition():
    """
    Simulates the concept of quantum superposition.

    In the physical world, this was demonstrated with a spinning coin. Here,
    we use a Hadamard gate on a qubit to put it into an equal superposition
    of states |0> and |1>. When measured, it randomly collapses to one of
    these states with roughly equal probability.
    """
    print("--- Part 1: Demonstrating Superposition ---\n")
    qc = QuantumCircuit(1, 1)
    print("Applying Hadamard gate to create superposition...")
    qc.h(0)
    qc.measure(0, 0)
    print("\nQuantum Circuit for Superposition:")
    print(qc.draw())
    simulator = AerSimulator()
    compiled_circuit = transpile(qc, simulator)
    shots = 1000
    result = simulator.run(compiled_circuit, shots=shots).result()
    counts = result.get_counts()
    print(f"\nResults after {shots} measurements:")
    print(counts)
    print("\nObservation: The results are approximately 50% '0' and 50% '1'.")
    print("This demonstrates that the qubit was in a superposition of both states before measurement.")
    plot_histogram(counts, title='Superposition Measurement Results')
    plt.show()
    print("\n" + "="*50 + "\n")

In [4]:
# Run the superposition demonstration
demonstrate_superposition()

--- Part 1: Demonstrating Superposition ---

Applying Hadamard gate to create superposition...

Quantum Circuit for Superposition:
     ┌───┐┌─┐
  q: ┤ H ├┤M├
     └───┘└╥┘
c: 1/══════╩═
           0 

Results after 1000 measurements:
{'1': 500, '0': 500}

Observation: The results are approximately 50% '0' and 50% '1'.
This demonstrates that the qubit was in a superposition of both states before measurement.




In [5]:
# CELL 4: PART 2 - ENTANGLEMENT
def demonstrate_entanglement():
    """
    Simulates quantum entanglement using a Bell state.

    This replaces the "entangled coins" exercise. We create a Bell state,
    where two qubits are linked. Measuring one instantly determines the
    state of the other, regardless of distance. When measured in the same
    basis, their outcomes are perfectly correlated.
    """
    print("--- Part 2: Demonstrating Entanglement (Bell State) ---\n")
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    qc.measure_all()
    print("Quantum Circuit for Bell State Entanglement:")
    print(qc.draw())
    simulator = AerSimulator()
    compiled_circuit = transpile(qc, simulator)
    result = simulator.run(compiled_circuit, shots=1000).result()
    counts = result.get_counts()
    print("\nResults of measuring the entangled pair:")
    print(counts)
    print("\nObservation: We only get '00' and '11' outcomes.")
    print("This perfect correlation is the hallmark of entanglement.")
    plot_histogram(counts, title='Entangled Bell State Measurement')
    plt.show()
    print("\n" + "="*50 + "\n")



--- Part 2: Demonstrating Entanglement (Bell State) ---

Quantum Circuit for Bell State Entanglement:
        ┌───┐      ░ ┌─┐   
   q_0: ┤ H ├──■───░─┤M├───
        └───┘┌─┴─┐ ░ └╥┘┌─┐
   q_1: ─────┤ X ├─░──╫─┤M├
             └───┘ ░  ║ └╥┘
   c: 2/══════════════╬══╬═
                      ║  ║ 
meas: 2/══════════════╩══╩═
                      0  1 

Results of measuring the entangled pair:
{'11 00': 487, '00 00': 513}

Observation: We only get '00' and '11' outcomes.
This perfect correlation is the hallmark of entanglement.




In [ ]:
# Run the entanglement demonstration
demonstrate_entanglement()

In [6]:
# CELL 5: PART 3 - BB84 QUANTUM KEY DISTRIBUTION
def simulate_bb84_protocol(n_bits=50):
    """
    Simulates the BB84 protocol for secure key exchange.

    This function simulates Alice sending qubits to Bob, and how they
    establish a shared secret key by comparing their randomly chosen bases.
    """
    print("--- Part 3: Simulating BB84 Quantum Key Distribution ---\n")
    simulator = AerSimulator()

    def alice_prepare_qubit(bit, basis):
        qc = QuantumCircuit(1, 1)
        if bit == 1: qc.x(0)
        if basis == 'X': qc.h(0)
        return qc

    def bob_measure_qubit(qc, basis):
        if basis == 'X': qc.h(0)
        qc.measure(0, 0)
        return qc

    alice_bits = [random.randint(0, 1) for _ in range(n_bits)]
    alice_bases = [random.choice(['Z', 'X']) for _ in range(n_bits)]
    print(f"Alice will send {n_bits} qubits.")
    bob_bases = [random.choice(['Z', 'X']) for _ in range(n_bits)]
    bob_results = []
    for i in range(n_bits):
        qc = alice_prepare_qubit(alice_bits[i], alice_bases[i])
        qc = bob_measure_qubit(qc, bob_bases[i])
        result = simulator.run(transpile(qc, simulator), shots=1).result()
        bob_results.append(int(list(result.get_counts().keys())[0]))

    sifted_key = []
    for i in range(n_bits):
        if alice_bases[i] == bob_bases[i] and alice_bits[i] == bob_results[i]:
            sifted_key.append(alice_bits[i])

    matching_bases = sum(1 for i in range(n_bits) if alice_bases[i] == bob_bases[i])
    print(f"\nOriginal number of bits sent: {n_bits}")
    print(f"Number of times bases matched: {matching_bases}")
    print(f"Final shared secret key length: {len(sifted_key)}")
    print(f"Theoretical efficiency: {matching_bases/n_bits:.2%}")
    print(f"Shared Key (first 20 bits): {sifted_key[:20]}")
    print("\n" + "="*50 + "\n")

In [7]:
# Run the BB84 simulation
simulate_bb84_protocol()

--- Part 3: Simulating BB84 Quantum Key Distribution ---

Alice will send 50 qubits.

Original number of bits sent: 50
Number of times bases matched: 27
Final shared secret key length: 27
Theoretical efficiency: 54.00%
Shared Key (first 20 bits): [0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0]




In [8]:
# CELL 6: PART 4 - BB84 WITH AN EAVESDROPPER (EVE)
def simulate_bb84_with_eve(n_bits=1000, eve_interception_prob=0.5):
    """
    Simulates the BB84 protocol with an eavesdropper, Eve.

    Eve intercepts qubits, measures them in a random basis, and resends
    them. This act of measurement introduces detectable errors in Alice
    and Bob's final key.
    """
    print(f"--- Part 4: BB84 with an Eavesdropper (Eve intercepts {eve_interception_prob:.0%} of qubits) ---\n")
    simulator = AerSimulator()

    def alice_prepare_qubit(bit, basis):
        qc = QuantumCircuit(1, 1)
        if bit == 1: qc.x(0)
        if basis == 'X': qc.h(0)
        return qc

    def measure_in_basis(qc, basis):
        if basis == 'X': qc.h(0)
        qc.measure(0, 0)
        return qc

    alice_bits = [random.randint(0, 1) for _ in range(n_bits)]
    alice_bases = [random.choice(['Z', 'X']) for _ in range(n_bits)]
    bob_bases = [random.choice(['Z', 'X']) for _ in range(n_bits)]
    bob_results = []

    for i in range(n_bits):
        qc = alice_prepare_qubit(alice_bits[i], alice_bases[i])
        if random.random() < eve_interception_prob:
            eve_basis = random.choice(['Z', 'X'])
            eve_qc = measure_in_basis(qc.copy(), eve_basis)
            eve_result = simulator.run(transpile(eve_qc, simulator), shots=1).result()
            eve_bit = int(list(eve_result.get_counts().keys())[0])
            qc = alice_prepare_qubit(eve_bit, eve_basis)

        bob_qc = measure_in_basis(qc, bob_bases[i])
        result = simulator.run(transpile(bob_qc, simulator), shots=1).result()
        bob_results.append(int(list(result.get_counts().keys())[0]))

    error_count = 0
    matching_bases_count = 0
    for i in range(n_bits):
        if alice_bases[i] == bob_bases[i]:
            matching_bases_count += 1
            if alice_bits[i] != bob_results[i]:
                error_count += 1

    error_rate = error_count / matching_bases_count if matching_bases_count > 0 else 0
    security_threshold = 0.11
    print(f"Simulation with {n_bits} bits and {eve_interception_prob:.0%} eavesdropping.")
    print(f"Number of matching bases: {matching_bases_count}")
    print(f"Errors detected: {error_count}")
    print(f"Quantum Bit Error Rate (QBER): {error_rate:.2%}")
    print(f"Security Threshold: {security_threshold:.2%}")

    if error_rate > security_threshold:
        print("\nResult: Eavesdropping DETECTED! The key is discarded.")
    else:
        print("\nResult: Communication appears secure. The key can be used.")
    print("\n" + "="*50 + "\n")

In [9]:
# Run simulations with Eve
simulate_bb84_with_eve(n_bits=1000, eve_interception_prob=0.5)
simulate_bb84_with_eve(n_bits=1000, eve_interception_prob=1.0)

--- Part 4: BB84 with an Eavesdropper (Eve intercepts 50% of qubits) ---

Simulation with 1000 bits and 50% eavesdropping.
Number of matching bases: 513
Errors detected: 65
Quantum Bit Error Rate (QBER): 12.67%
Security Threshold: 11.00%

Result: Eavesdropping DETECTED! The key is discarded.


--- Part 4: BB84 with an Eavesdropper (Eve intercepts 100% of qubits) ---

Simulation with 1000 bits and 100% eavesdropping.
Number of matching bases: 486
Errors detected: 119
Quantum Bit Error Rate (QBER): 24.49%
Security Threshold: 11.00%

Result: Eavesdropping DETECTED! The key is discarded.


